In [1]:
import numpy as np

data = np.genfromtxt(
    "../data/ETP_4_New_Data_Accidents.csv",
    delimiter=",",
    dtype=None,
    encoding="utf-8",
    names=True
)

print("Rows:", len(data))
print("Columns:", len(data.dtype.names))

Rows: 8116
Columns: 13


In [2]:
print("===== -1 VALUE CHECK =====")

for column in data.dtype.names:
    values = data[column]

    if np.issubdtype(values.dtype, np.number):
        count = np.sum(values == -1)
        print(f"{column:45} -> {count}")

===== -1 VALUE CHECK =====
Day_of_Week                                   -> 0
Accident_Location_A                           -> 0
Accident_Location_A_Chainage_km               -> 0
Accident_Location_A_Chainage_km_RoadSide      -> 0
Accident_Severity_C                           -> 0
Causes_D                                      -> 0
Road_Feature_E                                -> 0
Road_Condition_F                              -> 0
Weather_Conditions_H                          -> 0
Vehicle_Type_Involved_J_V1                    -> 0
Vehicle_Type_Involved_J_V2                    -> 6154


In [3]:
unique_data = np.unique(data)

print("Original rows:", len(data))
print("Unique rows:", len(unique_data))
print("Duplicate rows:", len(data) - len(unique_data))

Original rows: 8116
Unique rows: 4053
Duplicate rows: 4063


In [4]:
unique_rows, counts = np.unique(
    data,
    return_counts=True
)

print("===== DUPLICATE FREQUENCY =====")

print("Rows appearing once:",
      np.sum(counts == 1))

print("Rows appearing more than once:",
      np.sum(counts > 1))

print("Maximum repetitions of one row:",
      counts.max())

===== DUPLICATE FREQUENCY =====
Rows appearing once: 8
Rows appearing more than once: 4045
Maximum repetitions of one row: 6


In [5]:
duplicate_indices = np.where(counts > 1)[0]

print("\n===== EXAMPLE REPEATED RECORDS =====")

for index in duplicate_indices[:5]:
    print("\nRepeated", counts[index], "times:")
    print(unique_rows[index])


===== EXAMPLE REPEATED RECORDS =====

Repeated 2 times:
('1/1/2014', 3, '12:10', 2, 171.7, 1, 3, 2, 1, 1, 9, 1, -1)

Repeated 2 times:
('1/1/2015', 4, '17:10', 1, 502.0, 1, 3, 3, 4, 1, 1, 5, -1)

Repeated 2 times:
('1/1/2015', 4, '19:55', 2, 167.5, 1, 1, 1, 4, 1, 2, 5, 6)

Repeated 2 times:
('1/1/2015', 4, '8:20', 1, 495.9, 1, 3, 2, 4, 1, 1, 6, -1)

Repeated 2 times:
('1/1/2016', 5, '22:49', 2, 488.0, 1, 2, 4, 4, 1, 3, 6, -1)


In [6]:
duplicate_mask = counts > 1

duplicate_rows = np.sum(
    counts[duplicate_mask] * 1
)

rows_after_dedup = len(unique_rows)

print("Original records:", len(data))
print("Records after exact deduplication:", rows_after_dedup)
print("Records removed:", len(data) - rows_after_dedup)

print(
    "Percentage removed:",
    round((len(data) - rows_after_dedup) / len(data) * 100, 2),
    "%"
)

Original records: 8116
Records after exact deduplication: 4053
Records removed: 4063
Percentage removed: 50.06 %


In [7]:
original_severity = data["Accident_Severity_C"]

deduplicated_severity = unique_rows["Accident_Severity_C"]

print("===== SEVERITY: ORIGINAL vs DEDUPLICATED =====")

for severity in np.unique(original_severity):

    original_count = np.sum(original_severity == severity)
    dedup_count = np.sum(deduplicated_severity == severity)

    print(
        f"Severity {severity}: "
        f"{original_count} → {dedup_count}"
    )

===== SEVERITY: ORIGINAL vs DEDUPLICATED =====
Severity 1: 708 → 352
Severity 2: 2904 → 1450
Severity 3: 3088 → 1542
Severity 4: 1416 → 709


In [8]:
clean_data = unique_rows

print("Raw dataset:", data.shape)
print("Clean dataset:", clean_data.shape)

Raw dataset: (8116,)
Clean dataset: (4053,)


In [9]:
_, clean_counts = np.unique(
    clean_data,
    return_counts=True
)

print(
    "Duplicate rows remaining:",
    np.sum(clean_counts > 1)
)

Duplicate rows remaining: 0


In [10]:
dates = clean_data["Date"]

years = np.array([
    int(date.split("/")[2])
    for date in dates
])

months = np.array([
    int(date.split("/")[1])
    for date in dates
])

days = np.array([
    int(date.split("/")[0])
    for date in dates
])

print("Year range:", years.min(), "-", years.max())
print("Month range:", months.min(), "-", months.max())
print("Day range:", days.min(), "-", days.max())

Year range: 2003 - 2023
Month range: 1 - 12
Day range: 1 - 31


In [11]:
times = clean_data["Time_of_Accident"]

hours = np.array([
    int(time.split(":")[0])
    for time in times
])

minutes = np.array([
    int(time.split(":")[1])
    for time in times
])

print("Hour range:", hours.min(), "-", hours.max())
print("Minute range:", minutes.min(), "-", minutes.max())

Hour range: 0 - 23
Minute range: 0 - 59


In [12]:
print("===== DATE/TIME VALIDATION =====")

print("Invalid months:", np.sum((months < 1) | (months > 12)))
print("Invalid days:", np.sum((days < 1) | (days > 31)))
print("Invalid hours:", np.sum((hours < 0) | (hours > 23)))
print("Invalid minutes:", np.sum((minutes < 0) | (minutes > 59)))

===== DATE/TIME VALIDATION =====
Invalid months: 0
Invalid days: 0
Invalid hours: 0
Invalid minutes: 0


In [13]:
vehicle_v2 = clean_data["Vehicle_Type_Involved_J_V2"]

v2_missing_mask = vehicle_v2 == -1

print("No second vehicle:", np.sum(v2_missing_mask))
print(
    "Percentage:",
    round(np.mean(v2_missing_mask) * 100, 2),
    "%"
)

No second vehicle: 3075
Percentage: 75.87 %


In [14]:
vehicle_v2_clean = vehicle_v2.copy()

# Use 0 to represent "no second vehicle / not applicable"
vehicle_v2_clean[v2_missing_mask] = 0

print("Unique V2 values after cleaning:")
print(np.unique(vehicle_v2_clean))

Unique V2 values after cleaning:
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]


In [15]:
clean_features = np.column_stack([
    years,
    months,
    days,
    hours,
    minutes,
    clean_data["Day_of_Week"],
    clean_data["Accident_Location_A"],
    clean_data["Accident_Location_A_Chainage_km"],
    clean_data["Accident_Location_A_Chainage_km_RoadSide"],
    clean_data["Accident_Severity_C"],
    clean_data["Causes_D"],
    clean_data["Road_Feature_E"],
    clean_data["Road_Condition_F"],
    clean_data["Weather_Conditions_H"],
    clean_data["Vehicle_Type_Involved_J_V1"],
    vehicle_v2_clean
])

print("Shape:", clean_features.shape)
print("Dtype:", clean_features.dtype)

Shape: (4053, 16)
Dtype: float64


In [16]:
feature_names = np.array([
    "Year",
    "Month",
    "Day",
    "Hour",
    "Minute",
    "Day_of_Week",
    "Location",
    "Chainage_km",
    "RoadSide",
    "Severity",
    "Cause",
    "RoadFeature",
    "RoadCondition",
    "Weather",
    "Vehicle_V1",
    "Vehicle_V2"
])

print("Number of features:", len(feature_names))

for i, name in enumerate(feature_names):
    print(i, "->", name)

Number of features: 16
0 -> Year
1 -> Month
2 -> Day
3 -> Hour
4 -> Minute
5 -> Day_of_Week
6 -> Location
7 -> Chainage_km
8 -> RoadSide
9 -> Severity
10 -> Cause
11 -> RoadFeature
12 -> RoadCondition
13 -> Weather
14 -> Vehicle_V1
15 -> Vehicle_V2


In [17]:
print("===== FINAL CLEAN DATA VALIDATION =====")

print("Shape:", clean_features.shape)
print("NaN:", np.isnan(clean_features).sum())
print("Inf:", np.isinf(clean_features).sum())

print("\nFeature ranges:")
for i, name in enumerate(feature_names):
    print(
        f"{name:15} | "
        f"{clean_features[:, i].min():.2f} → "
        f"{clean_features[:, i].max():.2f}"
    )

===== FINAL CLEAN DATA VALIDATION =====
Shape: (4053, 16)
NaN: 0
Inf: 0

Feature ranges:
Year            | 2003.00 → 2023.00
Month           | 1.00 → 12.00
Day             | 1.00 → 31.00
Hour            | 0.00 → 23.00
Minute          | 0.00 → 59.00
Day_of_Week     | 1.00 → 7.00
Location        | 1.00 → 2.00
Chainage_km     | 0.20 → 741.39
RoadSide        | 1.00 → 2.00
Severity        | 1.00 → 4.00
Cause           | 1.00 → 8.00
RoadFeature     | 1.00 → 4.00
RoadCondition   | 1.00 → 8.00
Weather         | 1.00 → 12.00
Vehicle_V1      | 1.00 → 15.00
Vehicle_V2      | 0.00 → 15.00
